Step 1: Import Required Libraries

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras

Step 2: Load Audio File

In [ ]:
try:
    y, sr = librosa.load("audio.wav", duration=30)
except FileNotFoundError:
    print("Error: 'audio.wav' not found.")
    print("Please upload an audio file named 'audio.wav' to your Colab environment or provide the correct path to your audio file.")
    sr = 22050  # default sample rate
    duration = 30 # seconds
    y = np.zeros(int(sr * duration)) # Generate 30 seconds of silence
    print("Proceeding with 30 seconds of silence as dummy audio data.")

Error: 'audio.wav' not found.
Please upload an audio file named 'audio.wav' to your Colab environment or provide the correct path to your audio file.
Proceeding with 30 seconds of silence as dummy audio data.


/tmp/ipykernel_166/165919508.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load("audio.wav", duration=30)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Step 3: Extract Features (Mel Spectrogram)

In [ ]:
mel_spec = librosa.feature.melspectrogram(y=y, sr=sr)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

Step 4: Prepare Dataset

Convert all audio files into Mel Spectrograms

Resize spectrograms

Encode labels

Split into train & test sets

In [ ]:
num_samples = 10

num_features = mel_spec_db.shape[0] if 'mel_spec_db' in locals() else 128 # Default to 128 if not found
X = np.random.rand(num_samples, num_features) # Dummy features
y_labels = np.random.randint(0, 2, num_samples) # Dummy binary labels

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y_labels, test_size=0.2, random_state=42)

Step 5: Build CNN Model (Keras)

In [ ]:
model = keras.models.Sequential([
    keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 1)),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Step 6: Compile Model

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Step 7: Train Model

In [ ]:
# Re-define and re-compile the model to clear previous graph state if any
model = keras.models.Sequential([
    keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 1)),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Define the target input shape for the CNN
target_height = 128
target_width = 128
num_channels = 1

# Use the actual number of training and testing samples from the previous split
num_samples_train = X_train.shape[0]
num_samples_test = X_test.shape[0]

# Generate dummy features with the correct total number of elements per sample (128*128)
# In a real scenario, these would be your processed mel spectrograms, possibly resized to 128x128
X_train_reshaped = np.random.rand(num_samples_train, target_height * target_width)
X_test_reshaped = np.random.rand(num_samples_test, target_height * target_width)

# Reshape the dummy features into the 4D format expected by the CNN
X_train_reshaped = X_train_reshaped.reshape(-1, target_height, target_width, num_channels)
X_test_reshaped = X_test_reshaped.reshape(-1, target_height, target_width, num_channels)

# Use the existing y_train and y_test as they are labels and their shape is compatible
history = model.fit(X_train_reshaped, y_train, epochs=30, validation_data=(X_test_reshaped, y_test))
# --- End of fix (Demonstration Only) ---

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.0000e+00 - loss: 2.4590 - val_accuracy: 0.0000e+00 - val_loss: 9.5922
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step - accuracy: 0.8750 - loss: 1.1653 - val_accuracy: 0.0000e+00 - val_loss: 9.3055
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step - accuracy: 0.8750 - loss: 1.0780 - val_accuracy: 0.0000e+00 - val_loss: 6.4495
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step - accuracy: 0.8750 - loss: 0.6693 - val_accuracy: 0.0000e+00 - val_loss: 3.0149
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step - accuracy: 0.8750 - loss: 0.2255 - val_accuracy: 1.0000 - val_loss: 0.4916
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 290ms/step - accuracy: 0.1250 - loss: 0.7040 - val_accuracy: 0.0000e+00 - val_loss: 2.2722
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 1.0000 - loss: 0.1326 - val_accuracy: 0.0000e+00 - val_loss: 4.5539
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 0.8750 - loss: 0.2200 - val_accuracy: 0.00

Step 8: Evaluate Model

In [ ]:
test_loss, test_acc = model.evaluate(X_test_reshaped, y_test)
print("Test Accuracy:", test_acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.0000e+00 - loss: 6.0023
Test Accuracy: 0.0
